In [19]:
import os, json, random, time
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, classification_report
from collections import Counter
from tqdm import tqdm
from gensim.models import Word2Vec
from torchcrf import CRF
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(42); np.random.seed(42); random.seed(42)
print(f"Device: {device}")
if device.type == 'cuda': print(f"GPU: {torch.cuda.get_device_name(0)}")

import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid')


Device: cuda
GPU: NVIDIA GeForce GTX 1650 Ti


In [20]:
DATA_DIR = '.'
SAVE_DIR = './dl_all_models_bio_crf'

ASPECTS = ["CAMERA","FEATURES","PERFORMANCE","DESIGN","PRICE",
           "GENERAL","SCREEN","BATTERY","STORAGE","SER&ACC"]
SENTIMENTS = ["POSITIVE","NEUTRAL","NEGATIVE"]
LABEL_NAMES = [f"{a}#{s}" for a in ASPECTS for s in SENTIMENTS]
NUM_LABELS = 30
LABEL2ID = {n:i for i,n in enumerate(LABEL_NAMES)}

# BIO TAG SYSTEM: O + B-label + I-label = 61 tags
O_TAG = 0
BIO_TAGS = ['O']
for label in LABEL_NAMES:
    BIO_TAGS.append(f'B-{label}')  # Begin
    BIO_TAGS.append(f'I-{label}')  # Inside

NUM_TAGS = len(BIO_TAGS)  # 61
TAG2ID = {t:i for i,t in enumerate(BIO_TAGS)}

def get_bio_ids(label_name):
    return TAG2ID[f'B-{label_name}'], TAG2ID[f'I-{label_name}']

MAX_LEN = 128
MIN_FREQ = 2
W2V_DIM = 150
HIDDEN_DIM = 256
NUM_LAYERS = 2
DROPOUT = 0.3

BATCH_SIZE = 64
EPOCHS = 30
LR = 1e-3
PATIENCE = 7

print(f"✅ BIO Tag System: {NUM_TAGS} tags")

✅ BIO Tag System: 61 tags


In [21]:
# ============================================================
# CONVERT CHARACTER SPANS -> BIO TOKEN TAGS
# ============================================================
def text_to_bio_tags(text, spans, max_len):
    words = text.lower().split()[:max_len]
    positions = []
    pos = 0
    text_lower = text.lower()
    for w in words:
        idx = text_lower.find(w, pos)
        if idx == -1: idx = pos
        positions.append((idx, idx + len(w)))
        pos = idx + len(w)
    
    sorted_spans = sorted(spans, key=lambda s: s[1]-s[0])
    tags = [O_TAG] * max_len
    
    for start_char, end_char, label_str in sorted_spans:
        if label_str not in LABEL2ID: continue
        b_tag, i_tag = get_bio_ids(label_str)
        first_token = True
        for t_idx in range(len(words)):
            t_start, t_end = positions[t_idx]
            if t_start < end_char and t_end > start_char:
                if first_token:
                    tags[t_idx] = b_tag
                    first_token = False
                else:
                    tags[t_idx] = i_tag
    return tags, len(words)

def load_bio_data(filepath):
    texts, all_tags, sent_labels, lengths = [], [], [], []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            item = json.loads(line.strip())
            text = item['text']
            texts.append(text)
            spans = []
            sent = [0] * NUM_LABELS
            for s, e, label in item['labels']:
                if label in LABEL2ID:
                    sent[LABEL2ID[label]] = 1
                    spans.append((s, e, label))
            tags, length = text_to_bio_tags(text, spans, MAX_LEN)
            all_tags.append(tags)
            sent_labels.append(sent)
            lengths.append(length)
    return texts, all_tags, sent_labels, lengths

print("Loading & converting to BIO...")
train_texts, train_tags, train_sent, train_lens = load_bio_data(os.path.join(DATA_DIR, 'train.jsonl'))
dev_texts, dev_tags, dev_sent, dev_lens = load_bio_data(os.path.join(DATA_DIR, 'dev.jsonl'))
test_texts, test_tags, test_sent, test_lens = load_bio_data(os.path.join(DATA_DIR, 'test.jsonl'))
print(f"Train: {len(train_texts)} | Dev: {len(dev_texts)} | Test: {len(test_texts)}")

Loading & converting to BIO...
Train: 7785 | Dev: 1112 | Test: 2225


In [22]:
PAD_IDX = 0; UNK_IDX = 1
all_sentences = [t.lower().split() for t in train_texts + dev_texts + test_texts]
print(f"Training Word2Vec ({W2V_DIM}d)...")
w2v = Word2Vec(all_sentences, vector_size=W2V_DIM, window=5, min_count=MIN_FREQ, workers=4, epochs=20, sg=1, seed=42)

word2idx = {'<PAD>': PAD_IDX, '<UNK>': UNK_IDX}
for i, w in enumerate(w2v.wv.index_to_key): word2idx[w] = i + 2
VOCAB_SIZE = len(word2idx)

emb_matrix = np.random.normal(0, 0.1, (VOCAB_SIZE, W2V_DIM)).astype(np.float32)
emb_matrix[PAD_IDX] = 0
for w, idx in word2idx.items():
    if w in w2v.wv: emb_matrix[idx] = w2v.wv[w]

print(f"Vocab: {VOCAB_SIZE}")
def tokenize(text, max_len):
    words = text.lower().split()[:max_len]
    seq = [word2idx.get(w, UNK_IDX) for w in words]
    length = max(len(seq), 1)
    seq += [PAD_IDX] * (max_len - len(seq))
    return seq, length

Training Word2Vec (150d)...
Vocab: 7089


In [23]:
class BIODataset(Dataset):
    def __init__(self, texts, bio_tags, sent_labels, word2idx, max_len):
        self.sent_labels = torch.tensor(sent_labels, dtype=torch.float32)
        self.bio_tags = torch.tensor(bio_tags, dtype=torch.long)
        seqs, lens = [], []
        for t in texts:
            s, l = tokenize(t, max_len)
            seqs.append(s); lens.append(l)
        self.seqs = torch.tensor(seqs, dtype=torch.long)
        self.lens = torch.tensor(lens, dtype=torch.long)
        self.mask = torch.zeros(len(texts), max_len, dtype=torch.bool)
        for i, l in enumerate(lens):
            self.mask[i, :l] = True
    def __len__(self): return len(self.sent_labels)
    def __getitem__(self, i):
        return {'seq': self.seqs[i], 'len': self.lens[i], 'mask': self.mask[i],
                'tags': self.bio_tags[i], 'sent_labels': self.sent_labels[i]}

train_ds = BIODataset(train_texts, train_tags, train_sent, word2idx, MAX_LEN)
dev_ds = BIODataset(dev_texts, dev_tags, dev_sent, word2idx, MAX_LEN)
test_ds = BIODataset(test_texts, test_tags, test_sent, word2idx, MAX_LEN)

train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True)
dev_loader = DataLoader(dev_ds, BATCH_SIZE)
test_loader = DataLoader(test_ds, BATCH_SIZE)

In [24]:
# ============================================================
# SEQUENCE CRF & CNN CRF MODELS
# ============================================================

class SequenceCRF(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_dim, num_tags, pretrained_emb=None, n_layers=2, dropout=0.3, pad_idx=0, rnn_type='lstm', bidir=True):
        super().__init__()
        self.bidir = bidir
        rnn_out = hidden_dim * 2 if bidir else hidden_dim
        
        if pretrained_emb is not None:
            self.emb = nn.Embedding.from_pretrained(torch.FloatTensor(pretrained_emb), freeze=False, padding_idx=pad_idx)
        else:
            self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_idx)
        
        self.drop = nn.Dropout(dropout)
        
        if rnn_type == 'lstm': rnn_cls = nn.LSTM
        elif rnn_type == 'gru': rnn_cls = nn.GRU
        else: rnn_cls = nn.RNN
            
        self.rnn = rnn_cls(emb_dim, hidden_dim, n_layers, batch_first=True, dropout=dropout if n_layers > 1 else 0, bidirectional=bidir)
        self.hidden2tag = nn.Sequential(nn.Linear(rnn_out, rnn_out // 2), nn.ReLU(), nn.Dropout(dropout), nn.Linear(rnn_out // 2, num_tags))
        self.crf = CRF(num_tags, batch_first=True)
    
    def _get_emissions(self, seqs, lens):
        emb = self.drop(self.emb(seqs))
        packed = nn.utils.rnn.pack_padded_sequence(emb, lens.cpu().clamp(min=1), batch_first=True, enforce_sorted=False)
        output, _ = self.rnn(packed)
        output, _ = nn.utils.rnn.pad_packed_sequence(output, batch_first=True, total_length=seqs.size(1))
        return self.hidden2tag(self.drop(output))
    
    def forward(self, seqs, lens, mask, tags=None):
        emissions = self._get_emissions(seqs, lens)
        if tags is not None:
            loss = -self.crf(emissions, tags, mask=mask, reduction='mean')
            return {'loss': loss}
        else:
            best_tags = self.crf.decode(emissions, mask=mask)
            return {'tags': best_tags}

class CNNCRF(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_dim, num_tags, pretrained_emb=None, pad_idx=0, dropout=0.3):
        super().__init__()
        if pretrained_emb is not None:
            self.emb = nn.Embedding.from_pretrained(torch.FloatTensor(pretrained_emb), freeze=False, padding_idx=pad_idx)
        else:
            self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_idx)
        self.drop = nn.Dropout(dropout)
        
        self.convs = nn.ModuleList([
            nn.Conv1d(in_channels=emb_dim, out_channels=hidden_dim, kernel_size=3, padding=1),
            nn.Conv1d(in_channels=emb_dim, out_channels=hidden_dim, kernel_size=5, padding=2),
            nn.Conv1d(in_channels=emb_dim, out_channels=hidden_dim, kernel_size=7, padding=3)
        ])
        conv_out = hidden_dim * 3
        self.hidden2tag = nn.Sequential(nn.Linear(conv_out, conv_out // 2), nn.ReLU(), nn.Dropout(dropout), nn.Linear(conv_out // 2, num_tags))
        self.crf = CRF(num_tags, batch_first=True)
    
    def _get_emissions(self, seqs):
        emb = self.drop(self.emb(seqs)).transpose(1, 2)
        conv_outs = [torch.relu(conv(emb)) for conv in self.convs]
        out = torch.cat(conv_outs, dim=1).transpose(1, 2)
        return self.hidden2tag(self.drop(out))
        
    def forward(self, seqs, lens, mask, tags=None):
        emissions = self._get_emissions(seqs)
        if tags is not None:
            loss = -self.crf(emissions, tags, mask=mask, reduction='mean')
            return {'loss': loss}
        else:
            best_tags = self.crf.decode(emissions, mask=mask)
            return {'tags': best_tags}

In [25]:
def bio_tags_to_spans(tag_ids, max_tokens):
    spans = []
    current_label = None
    current_start = None
    for t in range(min(len(tag_ids), max_tokens)):
        tag_id = tag_ids[t]
        tag_name = BIO_TAGS[tag_id] if tag_id < len(BIO_TAGS) else 'O'
        if tag_name.startswith('B-'):
            if current_label is not None: spans.append((current_label, current_start, t))
            current_label = tag_name[2:]
            current_start = t
        elif tag_name.startswith('I-'):
            label = tag_name[2:]
            if current_label != label:
                if current_label is not None: spans.append((current_label, current_start, t))
                current_label = label
                current_start = t
        else:
            if current_label is not None:
                spans.append((current_label, current_start, t))
                current_label = None
    if current_label is not None:
        spans.append((current_label, current_start, len(tag_ids)))
    return spans

def bio_to_sentence_labels(tag_ids_list, lengths):
    sent_labels = []
    all_spans = []
    for tag_ids, length in zip(tag_ids_list, lengths):
        spans = bio_tags_to_spans(tag_ids, length)
        label_vec = [0] * NUM_LABELS
        for label_name, _, _ in spans:
            if label_name in LABEL2ID:
                label_vec[LABEL2ID[label_name]] = 1
        sent_labels.append(label_vec)
        all_spans.append(spans)
    return np.array(sent_labels), all_spans

In [26]:
def train_epoch(model, loader, optimizer):
    model.train()
    total = 0
    for b in tqdm(loader, leave=False):
        optimizer.zero_grad()
        out = model(b['seq'].to(device), b['len'], b['mask'].to(device), b['tags'].to(device))
        out['loss'].backward()
        nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
        total += out['loss'].item()
    return total / len(loader)

def predict_all(model, loader):
    model.eval()
    all_pred_tags, all_true_tags, all_sent_true, all_lengths = [], [], [], []
    total_loss = 0
    
    with torch.no_grad():
        for b in loader:
            out_loss = model(b['seq'].to(device), b['len'], b['mask'].to(device), b['tags'].to(device))
            total_loss += out_loss['loss'].item()
            out_pred = model(b['seq'].to(device), b['len'], b['mask'].to(device))
            for i in range(len(b['len'])):
                length = b['len'][i].item()
                pred_tags = out_pred['tags'][i][:length]
                true_tags = b['tags'][i][:length].tolist()
                all_pred_tags.append(pred_tags)
                all_true_tags.append(true_tags)
                all_lengths.append(length)
            all_sent_true.extend(b['sent_labels'].numpy())
            
    pred_sent, pred_spans = bio_to_sentence_labels(all_pred_tags, all_lengths)
    true_sent, true_spans = bio_to_sentence_labels(all_true_tags, all_lengths)
    correct, total_tokens = 0, 0
    for pt, tt in zip(all_pred_tags, all_true_tags):
        for p, t in zip(pt, tt):
            if p == t: correct += 1
            total_tokens += 1
    tok_acc = correct / max(1, total_tokens)
    
    return {'loss': total_loss / len(loader), 'pred_sent': pred_sent, 'true_sent': np.array(all_sent_true),
            'pred_spans': pred_spans, 'true_spans': true_spans, 'tok_acc': tok_acc}



In [27]:
# KHỞI TẠO LẠI VÀ LOAD TRỌNG SỐ TỪ FOLDER SAVE_DIR
import os

common = dict(
    vocab_size=VOCAB_SIZE, emb_dim=W2V_DIM, hidden_dim=HIDDEN_DIM,
    num_tags=NUM_TAGS, pretrained_emb=emb_matrix,
    n_layers=NUM_LAYERS, dropout=DROPOUT, pad_idx=PAD_IDX
)

models = {}

model_configs = {
    'TextCNN-CRF': lambda: CNNCRF(vocab_size=VOCAB_SIZE, emb_dim=W2V_DIM, hidden_dim=HIDDEN_DIM, num_tags=NUM_TAGS, pretrained_emb=emb_matrix, pad_idx=PAD_IDX, dropout=DROPOUT),
    'RNN-CRF': lambda: SequenceCRF(**common, rnn_type='rnn', bidir=False),
    'LSTM-CRF': lambda: SequenceCRF(**common, rnn_type='lstm', bidir=False),
    'BiLSTM-CRF': lambda: SequenceCRF(**common, rnn_type='lstm', bidir=True),
    'GRU-CRF': lambda: SequenceCRF(**common, rnn_type='gru', bidir=False),
    'BiGRU-CRF': lambda: SequenceCRF(**common, rnn_type='gru', bidir=True)
}

print("Đang load trọng số (weights) của các Baseline Models...")
for name, init_fn in model_configs.items():
    weight_path = os.path.join(SAVE_DIR, f'{name.lower().replace("-","_")}.pt')
    if os.path.exists(weight_path):
        m = init_fn().to(device)
        m.load_state_dict(torch.load(weight_path, map_location=device))
        models[name] = m
        print(f"✅ Loaded: {name}")
    else:
        print(f"❌ Không tìm thấy file trọng số cho: {name}")


Đang load trọng số (weights) của các Baseline Models...
✅ Loaded: TextCNN-CRF
✅ Loaded: RNN-CRF
✅ Loaded: LSTM-CRF
✅ Loaded: BiLSTM-CRF
✅ Loaded: GRU-CRF
✅ Loaded: BiGRU-CRF


In [28]:
import pandas as pd
import numpy as np
from collections import defaultdict
from IPython.display import display

def evaluate_paper_format(model, test_loader, aspects_list, sentiments_list):
    # Lấy dự đoán từ mô hình bằng hàm `predict_all` của bạn
    test_out = predict_all(model, test_loader)
    true_spans_list = test_out['true_spans']
    pred_spans_list = test_out['pred_spans']
    
    # 1. Khởi tạo bộ đếm (TP = True Positive, FP = False Positive, FN = False Negative)
    metrics = {
        'Aspect': {'tp': 0, 'fp': 0, 'fn': 0},
        'Polarity': {'tp': 0, 'fp': 0, 'fn': 0},
        'Aspect-Polarity': {'tp': 0, 'fp': 0, 'fn': 0}
    }
    
    aspect_counts = defaultdict(lambda: {'tp': 0, 'fp': 0, 'fn': 0})
    sentiment_counts = defaultdict(lambda: {'tp': 0, 'fp': 0, 'fn': 0})
    aspect_sentiment_counts = defaultdict(lambda: {'tp': 0, 'fp': 0, 'fn': 0})
    
    # 2. Bắt đầu đếm nhãn chuẩn xác 
    for true_spans, pred_spans in zip(true_spans_list, pred_spans_list):
        # Format đầu vào của span đang là: ('ASPECT#SENTIMENT', start, end)
        
        # 2a. Tách Sub-task: Aspect-Polarity (Match 100% Triple)
        t_ap = set([(s[0], s[1], s[2]) for s in true_spans])
        p_ap = set([(s[0], s[1], s[2]) for s in pred_spans])
        
        # 2b. Tách Sub-task: Aspect Only (Bỏ phần sentiment bằng split qua '#')
        t_a = set([(s[0].split('#')[0], s[1], s[2]) for s in true_spans])
        p_a = set([(s[0].split('#')[0], s[1], s[2]) for s in pred_spans])
        
        # 2c. Tách Sub-task: Polarity / Sentiment Only
        t_p = set([(s[0].split('#')[1], s[1], s[2]) for s in true_spans])
        p_p = set([(s[0].split('#')[1], s[1], s[2]) for s in pred_spans])
        
        def update_counts(true_set, pred_set, overall_dict, class_dicts, extract_key_fn=lambda x: x[0]):
            for span in true_set:
                key = extract_key_fn(span)
                if span in pred_set:
                    overall_dict['tp'] += 1; class_dicts[key]['tp'] += 1
                else:
                    overall_dict['fn'] += 1; class_dicts[key]['fn'] += 1
            for span in pred_set:
                if span not in true_set:
                    key = extract_key_fn(span)
                    overall_dict['fp'] += 1; class_dicts[key]['fp'] += 1

        update_counts(t_ap, p_ap, metrics['Aspect-Polarity'], aspect_sentiment_counts)
        update_counts(t_a, p_a, metrics['Aspect'], aspect_counts)
        update_counts(t_p, p_p, metrics['Polarity'], sentiment_counts)

    # ==========================
    # CÔNG THỨC HỖ TRỢ P, R, F1 (Đổi qua thang x100)
    # ==========================
    def calc_metrics(tp, fp, fn):
        p = tp / (tp + fp) if tp + fp > 0 else 0
        r = tp / (tp + fn) if tp + fn > 0 else 0
        f1 = 2 * p * r / (p + r) if p + r > 0 else 0
        return p * 100, r * 100, f1 * 100

    def calc_macro(counts_dict, keys):
        all_p, all_r, all_f1 = [], [], []
        for k in keys:
            c = counts_dict[k]
            p, r, f1 = calc_metrics(c['tp'], c['fp'], c['fn'])
            all_p.append(p); all_r.append(r); all_f1.append(f1)
        return np.mean(all_p), np.mean(all_r), np.mean(all_f1)

    # ==========================
    # TẠO CÁC BẢNG (DATAFRAMES) GIỐNG BÀI BÁO
    # ==========================
    
    # Table 3: The overall experimental results
    df3 = []
    for task in ['Aspect', 'Polarity', 'Aspect-Polarity']:
        c = metrics[task]
        p_mi, r_mi, f1_mi = calc_metrics(c['tp'], c['fp'], c['fn'])
        if task == 'Aspect':
            keys = aspects_list; counts = aspect_counts
        elif task == 'Polarity':
            keys = sentiments_list; counts = sentiment_counts
        else:
            keys = [f"{a}#{s}" for a in aspects_list for s in sentiments_list]; counts = aspect_sentiment_counts
            
        p_ma, r_ma, f1_ma = calc_macro(counts, keys)
        df3.append({'System': task, 
                    'PMicro': p_mi, 'RMicro': r_mi, 'F1Micro': f1_mi,
                    'PMacro': p_ma, 'RMacro': r_ma, 'F1Macro': f1_ma})
    df3 = pd.DataFrame(df3).round(2).set_index('System')

    # Table 4: Result per class for only aspect
    df4 = []
    for a in aspects_list:
        c = aspect_counts[a]
        p, r, f1 = calc_metrics(c['tp'], c['fp'], c['fn'])
        df4.append({'Aspect': a, 'Precision': p, 'Recall': r, 'F1-score': f1})
    df4 = pd.DataFrame(df4).round(2).set_index('Aspect')
    
    # Table 5: Sentiment evaluation
    df5 = []
    for s in sentiments_list:
        c = sentiment_counts[s]
        p, r, f1 = calc_metrics(c['tp'], c['fp'], c['fn'])
        df5.append({'Sentiment': s, 'Precision': p, 'Recall': r, 'F1-score': f1})
    df5 = pd.DataFrame(df5).round(2).set_index('Sentiment')
    
    # Table 6: F1-score per class cho aspect#polarity
    df6 = []
    for a in aspects_list:
        row = {'Aspect': a}
        for s in sentiments_list:
            c = aspect_sentiment_counts[f"{a}#{s}"]
            _, _, f1 = calc_metrics(c['tp'], c['fp'], c['fn'])
            row[s.capitalize()] = f1 # Capitalize để in thành Negative, Neutral, Positive
        df6.append(row)
    df6 = pd.DataFrame(df6).round(2).set_index('Aspect')
    
    # Output xuất ra giao diện trực quan
    print(f"\n{'='*95}\n📊 Table 3: The overall experimental results\n{'='*95}")
    display(df3)
    
    print(f"\n{'='*95}\n📊 Table 4: Result per class for only aspect label\n{'='*95}")
    display(df4)
    
    print(f"\n{'='*95}\n📊 Table 5: P, R, F1 for Sentiment level\n{'='*95}")
    display(df5)
    
    print(f"\n{'='*95}\n📊 Table 6: F1-score per class for aspect#polarity label\n{'='*95}")
    display(df6)

# ============================================
# 👉 ĐÁNH GIÁ CHO MODEL CÓ KẾT QUẢ TỐT NHẤT 
# ============================================
import json

try:
    # Đọc kết quả lưu từ quá trình training để biết model nào tốt nhất
    with open(os.path.join(SAVE_DIR, 'results.json'), 'r', encoding='utf-8') as f:
        results_data = json.load(f)
    
    # Tìm model có micro_f1 cao nhất
    best_model_name = max(results_data.keys(), key=lambda k: results_data[k].get('micro_f1', 0))
    print(f"\n\n🚀 BÁO CÁO CỰC ĐẠI CHO BEST MODEL: {best_model_name}")
    
    if best_model_name in models:
        evaluate_paper_format(models[best_model_name], test_loader, ASPECTS, SENTIMENTS)
    else:
        print(f"❌ Không tìm thấy model {best_model_name} đã load trong dictionary models.")
        
except FileNotFoundError:
    print(f"❌ Không tìm thấy file kết quả lưu tại {os.path.join(SAVE_DIR, 'results.json')}.")
    if len(models) > 0:
        first_model = list(models.keys())[-1]
        print(f"\nĐang chạy dự phòng cho: {first_model}")
        evaluate_paper_format(models[first_model], test_loader, ASPECTS, SENTIMENTS)




🚀 BÁO CÁO CỰC ĐẠI CHO BEST MODEL: BiGRU-CRF

📊 Table 3: The overall experimental results


,PMicro,RMicro,F1Micro,PMacro,RMacro,F1Macro
System,,,,,,
Aspect,54.56,53.38,53.97,51.95,49.52,50.55
Polarity,53.90,52.74,53.32,43.52,41.58,42.48
Aspect-Polarity,51.71,50.60,51.15,38.05,35.65,36.27



📊 Table 4: Result per class for only aspect label


,Precision,Recall,F1-score
Aspect,,,
CAMERA,60.33,60.42,60.37
FEATURES,45.08,45.19,45.13
PERFORMANCE,50.96,50.81,50.88
DESIGN,54.48,52.08,53.25
PRICE,36.09,32.42,34.16
GENERAL,58.86,56.52,57.67
SCREEN,49.50,54.55,51.90
BATTERY,62.72,62.03,62.37
STORAGE,47.83,32.35,38.60



📊 Table 5: P, R, F1 for Sentiment level


,Precision,Recall,F1-score
Sentiment,,,
POSITIVE,62.85,62.89,62.87
NEUTRAL,27.07,22.78,24.74
NEGATIVE,40.64,39.07,39.84



📊 Table 6: F1-score per class for aspect#polarity label


,Positive,Neutral,Negative
Aspect,,,
CAMERA,66.96,30.97,44.44
FEATURES,50.19,21.28,41.24
PERFORMANCE,61.86,11.85,32.04
DESIGN,59.50,8.33,24.56
PRICE,39.39,13.33,22.22
GENERAL,59.82,37.84,41.70
SCREEN,59.06,46.15,35.87
BATTERY,69.07,30.86,44.62
STORAGE,60.61,0.00,0.00
